# 01 - Data Preprocessing
This notebook loads the raw EV dataset, handles missing values, 
standardizes columns, engineers features, and saves the cleaned data.

In [1]:
# The required libraries to run the code

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
# Load dataset
df = pd.read_csv('Electric_Vehicle_Population_Data.csv')

print("Data Loaded!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns") # Specifies no. of rows and cols in the datset
print("\n Column Names & Data Types:") 
print(df.dtypes) # Specifies the datatypes of attributes or column values
print("\n Missing Values Per Column:")
print(df.isnull().sum()) # Specifies the count of missing values 
print("\n Statistical Summary:")
df.describe() # Gives statistical summary like count, mean, min/max and spread of the data

Data Loaded!
Shape: 271,113 rows × 16 columns

 Column Names & Data Types:
VIN (1-10)                                            object
County                                                object
City                                                  object
State                                                 object
Postal Code                                          float64
Model Year                                             int64
Make                                                  object
Model                                                 object
Electric Vehicle Type                                 object
Clean Alternative Fuel Vehicle (CAFV) Eligibility     object
Electric Range                                       float64
Legislative District                                 float64
DOL Vehicle ID                                         int64
Vehicle Location                                      object
Electric Utility                                      object
2020 Censu

,Postal Code,Model Year,Electric Range,Legislative District,DOL Vehicle ID,2020 Census Tract
count,271103.000000,271113.000000,271106.000000,270454.000000,2.711130e+05,2.711030e+05
mean,98176.923405,2021.995068,39.885639,28.813229,2.449143e+08,5.297202e+10
std,2576.510222,3.056730,78.764355,14.898449,6.365566e+07,1.632879e+09
min,1030.000000,1999.000000,0.000000,1.000000,4.385000e+03,1.001020e+09
25%,98052.000000,2021.000000,0.000000,17.000000,2.201288e+08,5.303301e+10
50%,98133.000000,2023.000000,0.000000,32.000000,2.619548e+08,5.303303e+10
75%,98382.000000,2024.000000,33.000000,42.000000,2.781037e+08,5.305394e+10
max,99577.000000,2026.000000,337.000000,49.000000,4.791150e+08,6.601095e+10


In [3]:
# ============================================================
#  Drop Unnecessary Columns & Fix Data Types
# ============================================================

# ---  Drop unnecessary columns ---
columns_to_drop = [
    'VIN (1-10)',          # unique ID, no analytical value
    'DOL Vehicle ID',      # unique ID, no analytical value
    'State',               # all values are 'WA', zero variation
    'Vehicle Location',    # raw GPS, we have County/City already
    '2020 Census Tract'    # too granular for county-level analysis
]

df = df.drop(columns=columns_to_drop)
print(f"Dropped {len(columns_to_drop)} columns")
print(f"Remaining columns: {df.shape[1]}")
print(df.columns.tolist())

# ---  Fix Data Types ---

# Postal Code → string (it's a label, not a number)
df['Postal Code'] = df['Postal Code'].astype(str).str.replace('.0', '', regex=False)

# Legislative District → string (it's a category, not a number)
df['Legislative District'] = df['Legislative District'].astype(str).str.replace('.0', '', regex=False)

# Electric Range → integer (whole miles, no decimals)
df['Electric Range'] = df['Electric Range'].fillna(0).astype(int)

# Model Year → already int, just confirm
print(f"\n Data Types Fixed!")
print(df.dtypes)

Dropped 5 columns
Remaining columns: 11
['County', 'City', 'Postal Code', 'Model Year', 'Make', 'Model', 'Electric Vehicle Type', 'Clean Alternative Fuel Vehicle (CAFV) Eligibility', 'Electric Range', 'Legislative District', 'Electric Utility']

 Data Types Fixed!
County                                               object
City                                                 object
Postal Code                                          object
Model Year                                            int64
Make                                                 object
Model                                                object
Electric Vehicle Type                                object
Clean Alternative Fuel Vehicle (CAFV) Eligibility    object
Electric Range                                        int64
Legislative District                                 object
Electric Utility                                     object
dtype: object


In [4]:
# ============================================================
# Handle Missing Values
# ============================================================

print("Missing values BEFORE cleaning:")
print(df.isnull().sum())

# ---  Drop rows where County/City/Postal Code is missing ---
# These are core location columns - without them the row is useless
df = df.dropna(subset=['County', 'City', 'Postal Code'])

# --- Handle Legislative District missing values ---
# 659 rows have missing district - we fill with 'Unknown'
# We don't drop them because the rest of the row is still valuable
df['Legislative District'] = df['Legislative District'].replace('nan', 'Unknown')

# --- Handle Electric Utility missing values ---
# Fill missing utility with 'Unknown'
df['Electric Utility'] = df['Electric Utility'].fillna('Unknown')

print("\nMissing values AFTER cleaning:")
print(df.isnull().sum())
print(f"\n Clean dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Rows removed: {271113 - df.shape[0]:,}")

Missing values BEFORE cleaning:
County                                               10
City                                                 10
Postal Code                                           0
Model Year                                            0
Make                                                  0
Model                                                 0
Electric Vehicle Type                                 0
Clean Alternative Fuel Vehicle (CAFV) Eligibility     0
Electric Range                                        0
Legislative District                                  0
Electric Utility                                     10
dtype: int64

Missing values AFTER cleaning:
County                                               0
City                                                 0
Postal Code                                          0
Model Year                                           0
Make                                                 0
Model                           

In [5]:
# ============================================================
#  Feature Engineering
# ============================================================

# --- Simplify CAFV Eligibility column ---
# Original values are long strings, let's simplify to short labels
df['CAFV_Category'] = df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].map({
    'Clean Alternative Fuel Vehicle Eligible': 'Eligible',
    'Not eligible due to low battery range': 'Not Eligible',
    'Eligibility unknown as battery range has not been researched': 'Unknown'
})

# ---  Simplify EV Type column ---
# Shorten long names to just BEV or PHEV
df['EV_Type'] = df['Electric Vehicle Type'].map({
    'Battery Electric Vehicle (BEV)': 'BEV',
    'Plug-in Hybrid Electric Vehicle (PHEV)': 'PHEV'
})

# --- Create Electric Range Category ---
# Group ranges into meaningful buckets
def categorize_range(r):
    if r == 0:
        return 'No Range'
    elif r <= 50:
        return 'Low'
    elif r <= 150:
        return 'Medium'
    else:
        return 'High'

df['Range_Category'] = df['Electric Range'].apply(categorize_range)

# --- Create Era column from Model Year ---
# Group years into meaningful periods
def categorize_era(year):
    if year <= 2012:
        return 'Early EV (≤2012)'
    elif year <= 2017:
        return 'Growth Era (2013-2017)'
    elif year <= 2021:
        return 'Mainstream (2018-2021)'
    else:
        return 'Modern (2022+)'

df['EV_Era'] = df['Model Year'].apply(categorize_era)

# --- Preview results ---
print("New features created!")
print("\nCAFV_Category values:")
print(df['CAFV_Category'].value_counts())
print("\nEV_Type values:")
print(df['EV_Type'].value_counts())
print("\nRange_Category values:")
print(df['Range_Category'].value_counts())
print("\nEV_Era values:")
print(df['EV_Era'].value_counts())

New features created!

CAFV_Category values:
CAFV_Category
Unknown         170809
Eligible         76205
Not Eligible     24089
Name: count, dtype: int64

EV_Type values:
EV_Type
BEV     216308
PHEV     54795
Name: count, dtype: int64

Range_Category values:
Range_Category
No Range    170816
Low          50589
High         33464
Medium       16234
Name: count, dtype: int64

EV_Era values:
EV_Era
Modern (2022+)            186800
Mainstream (2018-2021)     57202
Growth Era (2013-2017)     25056
Early EV (≤2012)            2045
Name: count, dtype: int64


In [6]:
# ============================================================
# County-Level Feature Aggregation
# ============================================================

# --- Create county-level summary ---
# This aggregates individual vehicle records into county profiles
county_features = df.groupby('County').agg(
    Total_EVs=('EV_Type', 'count'),
    BEV_Count=('EV_Type', lambda x: (x == 'BEV').sum()),
    PHEV_Count=('EV_Type', lambda x: (x == 'PHEV').sum()),
    Avg_Electric_Range=('Electric Range', 'mean'),
    Unique_Makes=('Make', 'nunique'),
    Unique_Models=('Model', 'nunique'),
    Most_Common_Make=('Make', lambda x: x.value_counts().index[0]),
    Avg_Model_Year=('Model Year', 'mean')
).reset_index()

# --- Calculate BEV and PHEV proportions ---
county_features['BEV_Proportion'] = (
    county_features['BEV_Count'] / county_features['Total_EVs']
).round(4)

county_features['PHEV_Proportion'] = (
    county_features['PHEV_Count'] / county_features['Total_EVs']
).round(4)

# --- Round numeric columns ---
county_features['Avg_Electric_Range'] = county_features['Avg_Electric_Range'].round(2)
county_features['Avg_Model_Year'] = county_features['Avg_Model_Year'].round(2)

print("County features created!")
print(f"Total counties: {county_features.shape[0]}")
print("\nPreview:")
county_features.head(10)

County features created!
Total counties: 246

Preview:


,County,Total_EVs,BEV_Count,PHEV_Count,Avg_Electric_Range,Unique_Makes,Unique_Models,Most_Common_Make,Avg_Model_Year,BEV_Proportion,PHEV_Proportion
0,Ada,1,1,0,0.00,1,1,TESLA,2023.00,1.0000,0.0000
1,Adams,118,91,27,47.59,18,37,TESLA,2021.75,0.7712,0.2288
2,Alachua,1,0,1,21.00,1,1,JEEP,2024.00,0.0000,1.0000
3,Alameda,9,8,1,2.33,4,6,TESLA,2022.56,0.8889,0.1111
4,Albemarle,1,1,0,0.00,1,1,TESLA,2024.00,1.0000,0.0000
5,Alexandria,8,7,1,75.38,3,6,TESLA,2022.25,0.8750,0.1250
6,Allegheny,2,2,0,0.00,2,2,MERCEDES-BENZ,2023.50,1.0000,0.0000
7,Allen,2,1,1,121.50,2,2,CHRYSLER,2017.50,0.5000,0.5000
8,Anchorage,2,1,1,7.00,2,2,BMW,2018.50,0.5000,0.5000
9,Anne Arundel,12,10,2,39.33,7,10,TESLA,2022.58,0.8333,0.1667


In [7]:
# ============================================================
# Data Validation
# ============================================================

# No duplicate removal needed - raw data has no true duplicates
# Apparent duplicates are caused by dropping identifier columns
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Validate Model Year
print("\nModel Year Validation:")
print(f"Min Model Year: {df['Model Year'].min()}")
print(f"Max Model Year: {df['Model Year'].max()}")
invalid_years = df[df['Model Year'] < 1990]
print(f"Vehicles with Model Year < 1990: {len(invalid_years)}")

# Validate Electric Range
print("\nElectric Range Validation:")
print(f"Min Range: {df['Electric Range'].min()}")
print(f"Max Range: {df['Electric Range'].max()}")
suspicious_range = df[df['Electric Range'] > 500]
print(f"Vehicles with Range > 500 miles: {len(suspicious_range)}")

# Validate EV Type
print("\nEV Type Validation:")
print(df['EV_Type'].value_counts())
invalid_ev = df[~df['EV_Type'].isin(['BEV', 'PHEV'])]
print(f"Invalid EV Types: {len(invalid_ev)}")

# County distribution check
print("\nCounty Distribution:")
print(f"Total unique counties: {df['County'].nunique()}")
print(f"Top 5 counties by EV count:")
print(df['County'].value_counts().head())

Dataset shape: 271,103 rows x 15 columns

Model Year Validation:
Min Model Year: 1999
Max Model Year: 2026
Vehicles with Model Year < 1990: 0

Electric Range Validation:
Min Range: 0
Max Range: 337
Vehicles with Range > 500 miles: 0

EV Type Validation:
EV_Type
BEV     216308
PHEV     54795
Name: count, dtype: int64
Invalid EV Types: 0

County Distribution:
Total unique counties: 246
Top 5 counties by EV count:
County
King         133815
Snohomish     33766
Pierce        22238
Clark         16694
Thurston       9911
Name: count, dtype: int64


In [8]:
# ============================================================
# Standardization of County Features for Clustering
# ============================================================

from sklearn.preprocessing import StandardScaler

# Select only numeric columns for standardization
numeric_cols = [
    'Total_EVs',
    'BEV_Count',
    'PHEV_Count',
    'Avg_Electric_Range',
    'Unique_Makes',
    'Unique_Models',
    'Avg_Model_Year',
    'BEV_Proportion',
    'PHEV_Proportion'
]

# Apply StandardScaler
scaler = StandardScaler()
county_scaled = county_features.copy()
county_scaled[numeric_cols] = scaler.fit_transform(county_features[numeric_cols])

print("Standardization complete!")
print("\nBefore standardization - Total_EVs stats:")
print(county_features['Total_EVs'].describe())
print("\nAfter standardization - Total_EVs stats:")
print(county_scaled['Total_EVs'].describe())

Standardization complete!

Before standardization - Total_EVs stats:
count       246.000000
mean       1102.044715
std        9004.311229
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max      133815.000000
Name: Total_EVs, dtype: float64

After standardization - Total_EVs stats:
count    2.460000e+02
mean     7.220963e-18
std      1.002039e+00
min     -1.225290e-01
25%     -1.225290e-01
50%     -1.224177e-01
75%     -1.220839e-01
max      1.476887e+01
Name: Total_EVs, dtype: float64


In [10]:
# ============================================================
# Save Cleaned Data & County Features
# ============================================================

# Save cleaned individual vehicle records
df.to_csv('outputs/results/cleaned_data.csv', index=False)
print(f"Saved cleaned_data.csv: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Save county-level features
county_features.to_csv('outputs/results/county_features.csv', index=False)
print(f"Saved county_features.csv: {county_features.shape[0]} rows x {county_features.shape[1]} columns")

# Save standardized county features for clustering
county_scaled.to_csv('outputs/results/county_scaled.csv', index=False)
print(f"Saved county_scaled.csv: {county_scaled.shape[0]} rows x {county_scaled.shape[1]} columns")

print("\n" + "="*50)
print("PREPROCESSING SUMMARY")
print("="*50)
print(f"Original dataset:     271,113 rows x 16 columns")
print(f"Cleaned dataset:      {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Rows removed:         {271113 - df.shape[0]:,}")
print(f"Columns removed:      {16 - df.shape[1]}")
print(f"New features added:   4 (CAFV_Category, EV_Type, Range_Category, EV_Era)")
print(f"County profiles:      {county_features.shape[0]} counties")
print("\nPreprocessing Complete! Ready for analysis.")

Saved cleaned_data.csv: 271,103 rows x 15 columns
Saved county_features.csv: 246 rows x 11 columns
Saved county_scaled.csv: 246 rows x 11 columns

PREPROCESSING SUMMARY
Original dataset:     271,113 rows x 16 columns
Cleaned dataset:      271,103 rows x 15 columns
Rows removed:         10
Columns removed:      1
New features added:   4 (CAFV_Category, EV_Type, Range_Category, EV_Era)
County profiles:      246 counties

Preprocessing Complete! Ready for analysis.
